# 04 — Train and compare delivery-time models
Compare a median baseline, Random Forest, and Extra Trees using an 80/20 holdout split. MAE is the main metric because it is interpretable in minutes.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib, json
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
P = ROOT/"data/processed"; MODELS = ROOT/"models"; REPORTS = ROOT/"reports"
MODELS.mkdir(exist_ok=True); REPORTS.mkdir(exist_ok=True)

df = pd.read_csv(P/"delivery_features.csv")
y = df.pop("target_delivery_minutes")
X = df
numeric = X.select_dtypes(include=["number"]).columns.tolist()
categorical = [c for c in X.columns if c not in numeric]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

def preprocessor():
    return ColumnTransformer([
        ("num", SimpleImputer(strategy="median", keep_empty_features=True), numeric),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent", keep_empty_features=True)),
            ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=5))
        ]), categorical)
    ])

models = {
    "Median baseline": DummyRegressor(strategy="median"),
    "Random Forest": Pipeline([("preprocess", preprocessor()), ("model",
        RandomForestRegressor(n_estimators=100, min_samples_leaf=2, n_jobs=-1, random_state=42))]),
    "Extra Trees": Pipeline([("preprocess", preprocessor()), ("model",
        ExtraTreesRegressor(n_estimators=100, min_samples_leaf=2, max_features=0.9, n_jobs=-1, random_state=42))])
}
results, fitted = [], {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = np.clip(model.predict(X_valid), 1, 180)
    results.append({"model": name,
                    "MAE_minutes": mean_absolute_error(y_valid, pred),
                    "RMSE_minutes": mean_squared_error(y_valid, pred)**0.5,
                    "R2": r2_score(y_valid, pred)})
    fitted[name] = model

results_df = pd.DataFrame(results).sort_values("MAE_minutes")
display(results_df.round(3))
results_df.to_csv(REPORTS/"delivery_model_comparison.csv", index=False)

,model,MAE_minutes,RMSE_minutes,R2
1,Random Forest,3.272,4.112,0.807
2,Extra Trees,3.458,4.361,0.783
0,Median baseline,7.549,9.367,-0.001


In [2]:
non_baseline = results_df[results_df["model"] != "Median baseline"]
best_name = non_baseline.iloc[0]["model"]
best_model = fitted[best_name]
best_model.fit(X, y)
joblib.dump(best_model, MODELS/"delivery_eta.joblib")

metadata = {
    "selected_model": best_name,
    "target": "delivery duration in minutes",
    "feature_columns": X.columns.tolist(),
    "numeric_features": numeric,
    "categorical_features": categorical,
    "training_rows": int(len(X)),
    "validation_split": "random 80/20; random_state=42",
    "note": "Offline prototype. Evaluate time-based validation and production drift before deployment."
}
(MODELS/"delivery_eta_metadata.json").write_text(json.dumps(metadata, indent=2))
print("Saved model to", MODELS/"delivery_eta.joblib")

Saved model to /Users/kakulrajput/Documents/food-ordering-delivery-ml/models/delivery_eta.joblib
